# Holiday Calendar (Idiomatic Notebook)

This notebook uses the well-supported `holidays` package to generate holiday dates with a concise, notebook-first workflow.

In [ ]:
# Notebook parameters
start_year = 2024
end_year = 2028
country = "US"
subdiv = None  # Example: "CA" for California

In [ ]:
# Install once per environment if needed
%pip -q install holidays ipywidgets

In [ ]:
import json
import pandas as pd
import holidays
import ipywidgets as widgets
from IPython.display import Javascript, display

In [ ]:
# Build a tidy holiday table
years = range(start_year, end_year + 1)
cal = holidays.country_holidays(country, subdiv=subdiv, years=years, observed=True)

df = (
    pd.Series(cal, name="Holiday")
    .rename_axis("Date")
    .sort_index()
    .reset_index()
)

df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year
df["Weekday"] = df["Date"].dt.day_name()

df.head(10)

In [ ]:
# Compact yearly summary
summary = (
    df.groupby("Year", as_index=False)
    .agg(Holiday_Count=("Holiday", "count"))
)
summary

In [ ]:
# Inspect one year
year = 2026
df.loc[df["Year"] == year, ["Date", "Weekday", "Holiday"]].reset_index(drop=True)

## Quick Validation Tests

Each test is a single code cell that shows expected vs actual values as notebook output.
Assertions enforce correctness without print-based status messages.

In [ ]:
# ApprovalTest type and helpers (separate approvals notebook with raw cells keyed by test id)
import html
import re
from pathlib import Path
from datetime import datetime, timezone

APPROVAL_LAST_ACTUAL = {}
APPROVAL_BASELINES = {}

def to_iso_records(frame):
    normalized = frame.copy()
    for col in normalized.columns:
        if pd.api.types.is_datetime64_any_dtype(normalized[col]):
            normalized[col] = normalized[col].dt.strftime("%Y-%m-%d")
    return normalized.to_dict("records")

def stable_records(records, sort_by):
    if not sort_by:
        return records
    frame = pd.DataFrame(records)
    return frame.sort_values(sort_by).reset_index(drop=True).to_dict("records")

def _normalize_cell_id(value):
    cleaned = re.sub(r"[^A-Za-z0-9_-]", "-", str(value)).strip("-")
    if not cleaned:
        cleaned = "approval-test"
    return cleaned[:64]

def _utc_now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def _resolve_tests_notebook_path():
    configured = globals().get("TEST_NOTEBOOK_PATH")
    if configured:
        return Path(configured).resolve()
    return (Path.cwd() / "holiday_calculator_idiomatic.ipynb").resolve()

def _resolve_approvals_notebook_path():
    configured = globals().get("APPROVALS_NOTEBOOK_PATH")
    if configured:
        return Path(configured).resolve()
    tests_nb = _resolve_tests_notebook_path()
    return tests_nb.parent / "__approvals__" / tests_nb.name

def _empty_notebook():
    return {
        "cells": [],
        "metadata": {
            "language_info": {"name": "python"},
            "approval_store": {"format": "raw-cell-per-test"},
        },
        "nbformat": 4,
        "nbformat_minor": 5,
    }

def _make_raw_approval_cell(test_id, value, created_at=None, updated_at=None):
    metadata = {
        "language": "raw",
        "approval_test_id": test_id,
    }
    if created_at:
        metadata["created_at"] = created_at
    if updated_at:
        metadata["updated_at"] = updated_at
    return {
        "cell_type": "raw",
        "id": _normalize_cell_id(test_id),
        "metadata": metadata,
        "source": [json.dumps(value, indent=2, ensure_ascii=True)],
    }

def _parse_cell_json_value(cell):
    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)
    try:
        return json.loads(source)
    except Exception:
        return None

def _read_approvals_notebook():
    path = _resolve_approvals_notebook_path()
    if not path.exists():
        return _empty_notebook()
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def _write_approvals_notebook(nb):
    path = _resolve_approvals_notebook_path()
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(nb, f, indent=4, ensure_ascii=True)

def _seed_initial_baselines_if_needed(nb):
    if nb.get("cells"):
        return nb
    seed = globals().get("INITIAL_APPROVAL_BASELINES", {})
    created_at = _utc_now_iso()
    for test_id, value in seed.items():
        nb["cells"].append(_make_raw_approval_cell(test_id, value, created_at=created_at))
    return nb

def _coerce_raw_cells_with_test_ids(nb):
    initial_keys = list((globals().get("INITIAL_APPROVAL_BASELINES", {}) or {}).keys())
    fallback_idx = 0
    changed = False
    normalized_cells = []

    for cell in nb.get("cells", []):
        value = _parse_cell_json_value(cell)
        if value is None:
            continue

        metadata = cell.get("metadata", {})
        test_id = metadata.get("approval_test_id")
        if not test_id:
            existing_id = str(cell.get("id", ""))
            if existing_id and not existing_id.startswith("#VSC-"):
                test_id = existing_id
        if not test_id and fallback_idx < len(initial_keys):
            test_id = initial_keys[fallback_idx]
            fallback_idx += 1
            changed = True
        if not test_id:
            continue

        created_at = metadata.get("created_at") or _utc_now_iso()
        updated_at = metadata.get("updated_at")
        normalized = _make_raw_approval_cell(
            test_id=test_id,
            value=value,
            created_at=created_at,
            updated_at=updated_at,
        )
        if cell.get("cell_type") != "raw" or metadata.get("approval_test_id") != test_id or cell.get("id") != normalized["id"]:
            changed = True
        normalized_cells.append(normalized)

    nb["cells"] = normalized_cells
    return nb, changed

def load_approval_baselines():
    nb = _read_approvals_notebook()
    nb = _seed_initial_baselines_if_needed(nb)
    nb, changed = _coerce_raw_cells_with_test_ids(nb)
    if changed:
        _write_approvals_notebook(nb)

    baselines = {}
    for cell in nb.get("cells", []):
        metadata = cell.get("metadata", {})
        test_id = metadata.get("approval_test_id")
        value = _parse_cell_json_value(cell)
        if test_id and value is not None:
            baselines[test_id] = value
    return baselines

def save_approval(test_id, value):
    nb = _read_approvals_notebook()
    nb = _seed_initial_baselines_if_needed(nb)
    nb, _ = _coerce_raw_cells_with_test_ids(nb)
    updated = False
    now = _utc_now_iso()

    for idx, cell in enumerate(nb.get("cells", [])):
        existing_id = cell.get("metadata", {}).get("approval_test_id")
        if existing_id == test_id:
            created_at = cell.get("metadata", {}).get("created_at") or now
            nb["cells"][idx] = _make_raw_approval_cell(
                test_id=test_id,
                value=value,
                created_at=created_at,
                updated_at=now,
            )
            updated = True
            break

    if not updated:
        nb.setdefault("cells", []).append(
            _make_raw_approval_cell(test_id=test_id, value=value, created_at=now)
        )

    _write_approvals_notebook(nb)

def clear_approval(test_id):
    nb = _read_approvals_notebook()
    nb = _seed_initial_baselines_if_needed(nb)
    nb, _ = _coerce_raw_cells_with_test_ids(nb)
    nb["cells"] = [
        cell for cell in nb.get("cells", [])
        if cell.get("metadata", {}).get("approval_test_id") != test_id
    ]
    _write_approvals_notebook(nb)

def clear_all_approvals():
    nb = _read_approvals_notebook()
    nb["cells"] = []
    _write_approvals_notebook(nb)

class ApprovalTest:
    def __init__(self, test_id, description, actual, approved=None, context=None):
        self.test_id = test_id
        self.description = description
        self.context = context or {"@vocab": "https://schema.org/"}
        self.actual = actual
        self.approved = approved

        if approved is None:
            self.passed = False
            self.status = "missing-approved"
            self.diff = {"approved": None, "actual": actual}
        elif approved == actual:
            self.passed = True
            self.status = "approved"
            self.diff = []
        else:
            self.passed = False
            self.status = "changed"
            self.diff = {"approved": approved, "actual": actual}

    def _jsonld_payload(self):
        return {
            "@context": self.context,
            "@type": "PropertyValue",
            "name": self.test_id,
            "description": self.description,
            "value": self.actual,
            "expectedValue": self.approved,
            "result": self.passed,
            "additionalProperty": [
                {"@type": "PropertyValue", "name": "status", "value": self.status},
                {"@type": "PropertyValue", "name": "diff", "value": self.diff},
                {"@type": "PropertyValue", "name": "approvalsNotebook", "value": str(_resolve_approvals_notebook_path())},
            ],
        }

    def _html_view(self):
        status_color = "#0f766e" if self.passed else "#b91c1c"
        border_color = "#14b8a6" if self.passed else "#ef4444"
        actual_text = html.escape(json.dumps(self.actual, indent=2, ensure_ascii=True))
        approved_text = html.escape(json.dumps(self.approved, indent=2, ensure_ascii=True))

        if self.passed:
            body = f"""
  <details style='margin-top:10px;' open>
    <summary><strong>Value</strong></summary>
    <pre style='white-space:pre-wrap; background:#f8fafc; padding:8px; border-radius:6px;'>{actual_text}</pre>
  </details>
"""
        else:
            body = f"""
  <details style='margin-top:10px;' open>
    <summary><strong>Approved</strong></summary>
    <pre style='white-space:pre-wrap; background:#f8fafc; padding:8px; border-radius:6px;'>{approved_text}</pre>
  </details>
  <details style='margin-top:10px;' open>
    <summary><strong>Actual</strong></summary>
    <pre style='white-space:pre-wrap; background:#f8fafc; padding:8px; border-radius:6px;'>{actual_text}</pre>
  </details>
"""

        return f"""
<div style='border:1px solid {border_color}; border-radius:10px; padding:12px; margin:8px 0; font-family:ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;'>
  <div style='display:flex; justify-content:space-between; align-items:center; gap:8px;'>
    <div>
      <strong>{html.escape(self.test_id)}</strong><br/>
      <span style='font-size:12px; color:#334155;'>{html.escape(self.description)}</span>
    </div>
    <span style='color:{status_color}; font-weight:700;'>{html.escape(self.status)}</span>
  </div>
  <div style='margin-top:8px; font-size:12px; color:#475569;'>
    Approvals notebook: <code>{html.escape(str(_resolve_approvals_notebook_path()))}</code>
  </div>
{body}
</div>
"""

    def _repr_mimebundle_(self, include=None, exclude=None):
        payload = self._jsonld_payload()
        data = {
            "application/ld+json": payload,
            "application/json": payload,
            "text/html": self._html_view(),
            "text/plain": json.dumps(payload, indent=2, ensure_ascii=True),
        }
        metadata = {
            "application/json": {"expanded": True},
            "application/ld+json": {
                "expanded": True,
                "approvalTest": {
                    "testId": self.test_id,
                    "status": self.status,
                    "approved": self.approved,
                },
            },
        }
        return data, metadata

def approval_action(action, test_id=None):
    if action == "approve":
        if test_id is None or test_id not in APPROVAL_LAST_ACTUAL:
            raise KeyError("No actual value registered for test")
        save_approval(test_id, APPROVAL_LAST_ACTUAL[test_id])
    elif action == "clear":
        if test_id is None:
            raise ValueError("clear requires test_id")
        clear_approval(test_id)
    elif action == "clear_all":
        clear_all_approvals()
    else:
        raise ValueError("action must be approve, clear, or clear_all")

    global APPROVAL_BASELINES
    APPROVAL_BASELINES = load_approval_baselines()

def approval_controls(test_id, passed):
    approve_btn = widgets.Button(description="Approve", button_style="success", icon="check")
    clear_btn = widgets.Button(description="Clear", button_style="warning", icon="trash")
    clear_all_btn = widgets.Button(description="Clear All", button_style="danger", icon="trash")
    passed_btn = widgets.Button(description="Passed", button_style="success", icon="check", disabled=True)
    status = widgets.HTML(value="<small>Approval actions write to approvals notebook raw cells.</small>")

    def on_approve(_):
        approval_action("approve", test_id)
        status.value = "<small><b>Approved.</b> Re-run this test cell to refresh status.</small>"

    def on_clear(_):
        approval_action("clear", test_id)
        status.value = "<small><b>Cleared.</b> Re-run this test cell to show missing-approved.</small>"

    def on_clear_all(_):
        approval_action("clear_all")
        status.value = "<small><b>All cleared.</b> Re-run test cells to refresh status.</small>"

    approve_btn.on_click(on_approve)
    clear_btn.on_click(on_clear)
    clear_all_btn.on_click(on_clear_all)

    buttons = [approve_btn, clear_btn, clear_all_btn]
    if passed:
        buttons.insert(0, passed_btn)

    return widgets.VBox([
        widgets.HBox(buttons),
        status,
    ])

def run_approval_test(test_id, description, actual, sort_by=None):
    global APPROVAL_BASELINES
    APPROVAL_BASELINES = load_approval_baselines()
    actual_sorted = stable_records(actual, sort_by) if isinstance(actual, list) else actual
    APPROVAL_LAST_ACTUAL[test_id] = actual_sorted
    approved = APPROVAL_BASELINES.get(test_id)
    return ApprovalTest(
        test_id=test_id,
        description=description,
        actual=actual_sorted,
        approved=approved,
    )

def show_approval_test(test_id, description, actual, sort_by=None):
    result = run_approval_test(test_id, description, actual, sort_by=sort_by)
    display(result)
    display(approval_controls(test_id, result.passed))
    return None

def approval_from_dataframe(test_id, description, actual_df, sort_by=None):
    actual_records = to_iso_records(actual_df)
    return show_approval_test(test_id, description, actual_records, sort_by=sort_by)

In [ ]:
# APPROVAL_CONFIG
TEST_NOTEBOOK_PATH = "/workspaces/coding/holiday_calculator_idiomatic.ipynb"
APPROVALS_NOTEBOOK_PATH = None  # Optional override path for approvals notebook
INITIAL_APPROVAL_BASELINES = {
    "required_columns_present": {
        "columns": ["Date", "Holiday", "Weekday", "Year"]
    },
    "year_boundaries": {
        "startYear": 2024,
        "endYear": 2028
    },
    "date_order_and_weekday_derivation": {
        "datesSorted": True,
        "weekdayMatchesDate": True
    },
    "known_holiday_spot_checks": [
        {"Date": "2024-01-01", "Expected": "New Year's Day"},
        {"Date": "2024-11-28", "Expected": "Thanksgiving Day"},
        {"Date": "2026-05-25", "Expected": "Memorial Day"}
    ]
}

In [ ]:
# Test 1: required columns are present (one-cell test)
actual_columns = sorted(df.columns.tolist())
required = ["Date", "Holiday", "Weekday", "Year"]
missing = [c for c in required if c not in actual_columns]
assert not missing, f"Missing columns: {missing}"

show_approval_test(
    test_id="required_columns_present",
    description="Dataframe includes required columns.",
    actual={"columns": actual_columns},
)

In [ ]:
# Test 2: year boundaries are correct (one-cell test)
actual_min = int(df["Year"].min())
actual_max = int(df["Year"].max())
assert actual_min == start_year, f"Start year mismatch: expected {start_year}, got {actual_min}"
assert actual_max == end_year, f"End year mismatch: expected {end_year}, got {actual_max}"

show_approval_test(
    test_id="year_boundaries",
    description="Observed year range matches configured notebook parameters.",
    actual={"startYear": actual_min, "endYear": actual_max},
)

In [ ]:
# Test 3: date ordering and weekday derivation (one-cell test)
is_sorted = bool(df["Date"].is_monotonic_increasing)
weekday_match = bool((df["Date"].dt.day_name() == df["Weekday"]).all())
assert is_sorted, "Dates are not sorted ascending"
assert weekday_match, "Weekday values do not match Date"

show_approval_test(
    test_id="date_order_and_weekday_derivation",
    description="Dates are sorted and weekday text matches derived weekday names.",
    actual={"datesSorted": is_sorted, "weekdayMatchesDate": weekday_match},
)

In [ ]:
# Test 4: known holiday spot checks (one-cell test)
known = pd.DataFrame([
    {"Date": pd.Timestamp("2024-01-01"), "Expected": "New Year's Day"},
    {"Date": pd.Timestamp("2024-11-28"), "Expected": "Thanksgiving Day"},
    {"Date": pd.Timestamp("2026-05-25"), "Expected": "Memorial Day"},
])

actual = (
    df.merge(known[["Date"]], on="Date", how="inner")
      .groupby("Date", as_index=False)
      .agg(Expected=("Holiday", lambda s: " | ".join(sorted(set(s)))))
)
actual_records = to_iso_records(actual)

show_approval_test(
    test_id="known_holiday_spot_checks",
    description="Known holiday checkpoints match expected names for specific dates.",
    actual=actual_records,
    sort_by=["Date", "Expected"],
)

## Concise Approval Tests (Visible Approved vs Actual)

This style keeps approvals directly in notebook cells.
Each test cell displays Approved and Actual values, then fails with a focused diff when they differ.

### Notebook Testing Patterns In Practice

- ipytest: run pytest directly inside notebook cells via magics.
- nbval or pytest-notebook: rerun whole notebooks and compare stored outputs, usually in CI.
- testbook: write tests in separate Python files against notebook code.
- ApprovalTests.Python: classic approved vs received workflow with diff tools.

This notebook uses per-cell approved tables so each test visibly shows Approved and Actual values in one place.

In [ ]:
# Helper note: JSON-LD helpers are defined above the tests to support linear top-to-bottom execution.
# This cell is intentionally left minimal to avoid redefining hooks later in the notebook.

In [ ]:
# Approval test 1: full 2026 holiday list (approved baseline stored in storage cell)
actual_2026 = df.loc[df["Year"] == 2026, ["Date", "Holiday"]]
approval_from_dataframe(
    test_id="approval_us_2026_holidays",
    description="Federal holidays for 2026 (including observed).",
    actual_df=actual_2026,
    sort_by=["Date", "Holiday"],
)

In [ ]:
# Approval test 2: Independence Day observed behavior in 2026
actual_observed = df.loc[
    (df["Date"].between(pd.Timestamp("2026-07-03"), pd.Timestamp("2026-07-04")))
    & (df["Holiday"].str.contains("Independence Day")),
    ["Date", "Holiday"],
]
approval_from_dataframe(
    test_id="approval_independence_day_observed_2026",
    description="Observed Independence Day appears on Friday when July 4 is Saturday.",
    actual_df=actual_observed,
    sort_by=["Date", "Holiday"],
)